# Phase 3: Rhetorical Roles: GQA decoder

This is the **Rhetorical Roles** part of the full `gqa` notebook (`notebooks/CS781_gqa_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

## Rhetorical Role Prediction

In [ ]:
import os, json, glob, shutil, torch
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
shutil.copy(f"{DRIVE_PROJECT_DIR}/legal_bpe_32k_FINAL.json", "legal_bpe_32k_FINAL.json")
tokenizer = ByteBPE.from_json("legal_bpe_32k_FINAL.json")
assert tokenizer.get_vocab_size() == 32000
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# HF_TOKEN = "..."   # set it as before (the Phase-2 setup cell used it)
print("device:", device, "| vocab:", tokenizer.get_vocab_size())
for p in sorted(glob.glob(f"{DRIVE_PROJECT_DIR}/**/*.pt", recursive=True)): print(" ", p)

device: cuda | vocab: 32000
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/DeepSeek/deepseek_mla_85m_pretrained.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt
  /content/drive/MyDrive/CS781_Capstone_Proj

In [ ]:
import os, json, math, time, random, pickle, collections
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from huggingface_hub import hf_hub_download
for n in ["tokenizer", "device", "HF_TOKEN", "HF_DATASET_REPO"]: assert n in globals(), f"'{n}' missing, run the setup cell"

P3_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_3"; os.makedirs(P3_DIR + "/submission", exist_ok=True)
WINDOW, MAX_SENT_TOK = 1024, 128
USE_CLASS_WEIGHTS, CLASS_W_CLIP = True, 10.0
PAD_ID = tokenizer.token_to_id("[PAD]"); NL_IDS = tokenizer.encode("\n").ids
def amp(): return torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)

# ---------------- data ----------------
def _load(f): return json.load(open(hf_hub_download(HF_DATASET_REPO, f"Phase_3_RR/{f}", repo_type="dataset", token=HF_TOKEN), encoding="utf-8"))
L2I = _load("Train/label_vocab.json"); LABELS = sorted(L2I, key=L2I.get); NC = len(LABELS)

def norm_docs(raw, domain):
    items = list(raw.items()) if isinstance(raw, dict) else raw
    docs = []
    for it in items:
        did, v = (it if isinstance(it, tuple) else (it["id"], it))
        text = v if isinstance(v, list) else v["text"]
        labels = v.get("labels") if isinstance(v, dict) else None
        d = dict(id=did, domain=domain, text=text, labels=labels)
        d["sent_ids"] = [tokenizer.encode(s.strip()).ids[:MAX_SENT_TOK] + NL_IDS for s in text]
        d["y"] = [L2I[l] for l in labels] if labels is not None else None
        if labels is not None: assert len(labels) == len(text)
        docs.append(d)
    return docs

train_docs = norm_docs(_load("Train/CL_train.json"), "CL") + norm_docs(_load("Train/IT_train.json"), "IT")
val_docs   = norm_docs(_load("Train/CL_dev.json"), "CL")   + norm_docs(_load("Train/IT_dev.json"), "IT")
test_docs  = norm_docs(_load("Test/CL_test.json"), "CL")   + norm_docs(_load("Test/IT_test.json"), "IT")
assert len({d["id"] for d in test_docs}) == len(test_docs), "test doc ids are not unique across domains"
for nm, ds in [("train", train_docs), ("val", val_docs), ("test", test_docs)]:
    print(f"{nm:5s} docs={len(ds)} sents={sum(len(d['text']) for d in ds)} "
          f"CL={sum(d['domain']=='CL' for d in ds)} IT={sum(d['domain']=='IT' for d in ds)}")

cnt = np.bincount([y for d in train_docs for y in d["y"]], minlength=NC).astype(float)
CLASS_W = torch.tensor(np.clip(np.sqrt(cnt.sum() / (NC * np.maximum(cnt, 1))), 0, CLASS_W_CLIP), dtype=torch.float32, device=device)
print("class weights:", {LABELS[i]: round(float(CLASS_W[i]), 2) for i in range(NC)})

# ---------------- context modes / windows ----------------
CTX = {"sentence": dict(max_sents=1, stride_frac=0.5),        # each sentence alone
       "local8":   dict(max_sents=8, stride_frac=0.5),        # up to 8 consecutive sentences
       "document": dict(max_sents=10**9, stride_frac=0.75)}   # as many sentences as fit in 1024 tokens
BW = {"sentence": 64, "local8": 32, "document": 8}            # windows per batch

def make_windows(sent_ids, max_sents, stride_frac, max_tokens=WINDOW):
    n, lens, wins, a = len(sent_ids), [len(x) for x in sent_ids], [], 0
    while True:
        b, tot = a, 0
        while b < n and tot + lens[b] <= max_tokens and (b - a) < max_sents: tot += lens[b]; b += 1
        wins.append((a, b))
        if b >= n: break
        c, acc, target = a, 0, max(1, int(stride_frac * tot))
        while c < b and acc < target: acc += lens[c]; c += 1
        a = max(c, a + 1)
    return wins

def all_windows(docs, mode): return [(di, a, b) for di, d in enumerate(docs) for a, b in make_windows(d["sent_ids"], **CTX[mode])]

def collate(wins, docs):
    seqs, spans, ys = [], [], []
    for di, a, b in wins:
        d = docs[di]; ids, sp = [], []
        for k in range(a, b): s = len(ids); ids += d["sent_ids"][k]; sp.append((s, len(ids)))
        seqs.append(ids); spans.append(sp)
        if d["y"] is not None: ys += d["y"][a:b]
    L = max(len(s) for s in seqs)
    x = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    for j, s in enumerate(seqs): x[j, :len(s)] = torch.tensor(s)
    return x, spans, (torch.tensor(ys) if ys else None)

# ---------------- model: backbone + sentence pooling + head ----------------
def pool(H, spans):
    jw, st, en = [], [], []
    for j, sp in enumerate(spans):
        for s, e in sp: jw.append(j); st.append(s); en.append(e)
    jw, st, en = [torch.tensor(t, device=H.device) for t in (jw, st, en)]
    last = H[jw, en - 1]                                                   # delimiter token: has seen the whole sentence
    cs = torch.cat([torch.zeros_like(H[:, :1]).float(), H.float().cumsum(1)], dim=1)
    mean = ((cs[jw, en] - cs[jw, st]) / (en - st).unsqueeze(1)).to(H.dtype)  # mean over the sentence's tokens
    return torch.cat([last, mean], -1)

class RRModel(nn.Module):
    """Decoder backbone (causal) -> per-sentence features [last token ; mean] -> linear head."""
    def __init__(self, lm, d_model, n_labels=NC, dropout=0.1):
        super().__init__()
        self.lm, self.drop, self.head = lm, nn.Dropout(dropout), nn.Linear(2 * d_model, n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
    def hidden(self, ids):
        lm = self.lm; x = lm.token_embedding(ids)
        for layer in lm.layers: x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
        return lm.norm(x)
    def forward(self, ids, spans): return self.head(self.drop(pool(self.hidden(ids), spans)))

# ---------------- metrics (own implementation) ----------------
def rr_scores(gold, pred):
    g, p = np.array(gold), np.array(pred); labs = sorted(set(g.tolist()) | set(p.tolist())); per = {}
    for l in labs:
        tp = int(((g == l) & (p == l)).sum()); fp = int(((g != l) & (p == l)).sum()); fn = int(((g == l) & (p != l)).sum())
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[LABELS[l]] = dict(p=pr, r=rc, f1=2 * pr * rc / (pr + rc) if pr + rc else 0.0, support=int((g == l).sum()))
    m = lambda k: float(np.mean([v[k] for v in per.values()]))
    return dict(macro_f1=m("f1"), macro_p=m("p"), macro_r=m("r"), acc=float((g == p).mean()), per=per)

# ---------------- prediction ----------------
@torch.no_grad()
def predict_probs(model, docs, mode):
    model.eval(); bw = BW[mode]; out = {}
    for d in docs:
        n = len(d["sent_ids"]); P = torch.zeros(n, NC, device=device); C = torch.zeros(n, device=device)
        wins = [(0, a, b) for a, b in make_windows(d["sent_ids"], **CTX[mode])]
        for i in range(0, len(wins), bw):
            chunk = wins[i:i + bw]; x, spans, _ = collate(chunk, [d])
            with amp(): lg = model(x.to(device), spans)
            p = F.softmax(lg.float(), -1); k = 0
            for _, a, b in chunk: P[a:b] += p[k:k + b - a]; C[a:b] += 1; k += b - a
        out[d["id"]] = (P / C.unsqueeze(1)).cpu().numpy()
    return out

def evaluate(model, docs, mode):
    P = predict_probs(model, docs, mode)
    G = {dm: ([], []) for dm in ("CL", "IT", "ALL")}
    for d in docs:
        pr = P[d["id"]].argmax(1).tolist()
        for key in (d["domain"], "ALL"): G[key][0].extend(d["y"]); G[key][1].extend(pr)
    res = rr_scores(*G["ALL"]); res["by_domain"] = {dm: rr_scores(*G[dm]) for dm in ("CL", "IT") if G[dm][0]}
    return res, P

def print_rr(res):
    print(f"ALL  macro-F1 {res['macro_f1']:.4f} | P {res['macro_p']:.4f} | R {res['macro_r']:.4f} | acc {res['acc']:.4f}")
    for dm, r in res["by_domain"].items(): print(f"  {dm}  macro-F1 {r['macro_f1']:.4f} | P {r['macro_p']:.4f} | R {r['macro_r']:.4f} | acc {r['acc']:.4f}")

def write_test_predictions(model, name, mode):
    P = predict_probs(model, test_docs, mode)
    allp = {d["id"]: [LABELS[i] for i in P[d["id"]].argmax(1)] for d in test_docs}
    for d in test_docs: assert len(allp[d["id"]]) == len(d["text"])
    for suffix, sel in [("all", allp), ("CL", {d["id"]: allp[d["id"]] for d in test_docs if d["domain"] == "CL"}),
                        ("IT", {d["id"]: allp[d["id"]] for d in test_docs if d["domain"] == "IT"})]:
        json.dump(sel, open(f"{P3_DIR}/prediction_{name}_{suffix}.json", "w"), ensure_ascii=False)
    pickle.dump(P, open(f"{P3_DIR}/probs_test_{name}.pkl", "wb"))
    print(f"Saved prediction_{name}_{{all,CL,IT}}.json ({len(allp)} docs, {sum(map(len, allp.values()))} sentences)")

# ---------------- trainer (optional distillation from a teacher RRModel) ----------------
def train_rr(model, name, mode="document", teacher=None, alpha=0.0, T=2.0, epochs=10, patience=3,
             backbone_lr=1e-4, head_lr=1e-3, bw=None, weight_decay=0.01, warmup_frac=0.06, debug=False, seed=1):
    random.seed(seed); torch.manual_seed(seed); bw = bw or BW[mode]
    tw = all_windows(train_docs, mode); vdocs = val_docs
    if debug:
        epochs, name = 1, name + "_debug"; tw = tw[:bw * 4]
        vdocs = [next(d for d in val_docs if d["domain"] == dm) for dm in ("CL", "IT")]
    model.to(device)
    if teacher is not None:
        teacher.to(device).eval()
        for p in teacher.parameters(): p.requires_grad_(False)
    groups = {}
    for n_, p in model.named_parameters():
        if p.requires_grad: groups.setdefault((n_.startswith("head."), p.dim() >= 2), []).append(p)
    opt = torch.optim.AdamW([{"params": ps, "lr": head_lr if h else backbone_lr, "weight_decay": weight_decay if dec else 0.0}
                             for (h, dec), ps in groups.items()], betas=(0.9, 0.98))
    spe = math.ceil(len(tw) / bw); total = spe * epochs; warm = max(1, int(warmup_frac * total))
    sched = LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{name}] mode={mode} windows={len(tw)} trainable={n_train:,} {'KD alpha=%.2f T=%.1f' % (alpha, T) if teacher is not None else ''}")
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    best, best_state, bad, hist = -1.0, None, 0, []
    cw = CLASS_W if USE_CLASS_WEIGHTS else None
    for ep in range(epochs):
        model.train(); random.shuffle(tw); tl, nb = 0.0, 0
        for i in range(0, len(tw), bw):
            x, spans, y = collate(tw[i:i + bw], train_docs); x, y = x.to(device), y.to(device)
            with amp():
                lg = model(x, spans).float()
                loss = F.cross_entropy(lg, y, weight=cw)
                if teacher is not None:
                    with torch.no_grad(): tlg = teacher(x, spans).float()
                    kd = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(tlg / T, -1), reduction="batchmean") * T * T
                    loss = (1 - alpha) * loss + alpha * kd
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt.step(); sched.step(); tl += loss.item(); nb += 1
        res, _ = evaluate(model, vdocs, mode)
        hist.append(dict(epoch=ep + 1, loss=tl / nb, macro_f1=res["macro_f1"], **{f"f1_{k}": v["macro_f1"] for k, v in res["by_domain"].items()}))
        print(f"  ep{ep+1:2d} loss {tl/nb:.4f} | val macro-F1 {res['macro_f1']:.4f} "
              + " ".join(f"{k} {v['macro_f1']:.3f}" for k, v in res["by_domain"].items()) + f" | {(time.time()-t0)/60:.1f} min")
        if res["macro_f1"] > best:
            best, bad = res["macro_f1"], 0
            names = {n_ for n_, p in model.named_parameters() if p.requires_grad}
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}
        else:
            bad += 1
            if bad >= patience: print("  early stop"); break
    model.load_state_dict(best_state, strict=False)
    path = f"{P3_DIR}/rr_{name}_best.pt"; torch.save(best_state, path)
    res, P = evaluate(model, vdocs, mode)
    print(f"[{name}] BEST on dev:"); print_rr(res)
    return dict(res=res, history=hist, best_path=path, minutes=(time.time() - t0) / 60,
                peak_gb=torch.cuda.max_memory_allocated() / 1e9, trainable_params=n_train, val_probs=P)

# ---------------- sanity checks (no model needed) ----------------
for mode in CTX:
    W = all_windows(train_docs, mode)
    cover = collections.defaultdict(set)
    for di, a, b in W: cover[di].update(range(a, b))
    assert all(len(cover[di]) == len(d["sent_ids"]) for di, d in enumerate(train_docs)), f"{mode}: some sentences not covered"
    mx = max(sum(len(train_docs[di]["sent_ids"][k]) for k in range(a, b)) for di, a, b in W)
    print(f"{mode:9s} windows={len(W):5d} | every sentence covered | longest window {mx} tokens (<= {WINDOW})")
    assert mx <= WINDOW
tt = sorted(sum(map(len, d["sent_ids"])) for d in test_docs)
print("test tokens/doc median", tt[len(tt) // 2], "max", tt[-1], "| val labels:", collections.Counter(LABELS[y] for d in val_docs for y in d["y"]).most_common(4), "...")
print("✅ Phase-3 toolkit ready")

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

train docs=80 sents=16845 CL=40 IT=40
val   docs=10 sents=2142 CL=5 IT=5
test  docs=10 sents=2197 CL=5 IT=5
class weights: {'Fact': 0.56, 'Issue': 3.48, 'ArgumentPetitioner': 0.9, 'ArgumentRespondent': 0.98, 'Statute': 1.09, 'Dissent': 2.14, 'PrecedentReliedUpon': 0.82, 'PrecedentNotReliedUpon': 1.51, 'PrecedentOverruled': 10.0, 'RulingByLowerCourt': 1.22, 'RatioOfTheDecision': 0.55, 'RulingByPresentCourt': 2.05, 'None': 2.09}
sentence  windows=16845 | every sentence covered | longest window 129 tokens (<= 1024)
local8    windows= 3694 | every sentence covered | longest window 825 tokens (<= 1024)
document  windows=  892 | every sentence covered | longest window 1024 tokens (<= 1024)
test tokens/doc median 7457 max 35993 | val labels: [('Fact', 444), ('RatioOfTheDecision', 403), ('ArgumentPetitioner', 400), ('ArgumentRespondent', 272)] ...
✅ Phase-3 toolkit ready


In [ ]:
DEBUG_MODE = False
assert "CustomDecoderLM" in globals()
TEACHER_CKPT = f"{DRIVE_PROJECT_DIR}/gqa_checkpoints/gqa_best.pt"
assert os.path.exists(TEACHER_CKPT), TEACHER_CKPT

def build_gqa_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True)
    ck = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)      # strict: pretrained Phase-1 weights
    return RRModel(lm_, 768, NC, dropout).to(device)

RR_RESULTS = {}
MODES = ["document"] if DEBUG_MODE else ["document", "local8", "sentence"]
for mode in MODES:
    model = build_gqa_rr()
    out = train_rr(model, f"gqa_full_{mode}", mode=mode, epochs=10, patience=3,
                   backbone_lr=1e-4, head_lr=1e-3, debug=DEBUG_MODE)
    RR_RESULTS[mode] = out
    if not DEBUG_MODE:
        write_test_predictions(model, f"gqa_full_{mode}", mode)
    del model; torch.cuda.empty_cache()

print(f"\n{'mode':10s} {'F1(all)':>8s} {'CL':>7s} {'IT':>7s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for m, o in RR_RESULTS.items():
    r = o["res"]; bd = r["by_domain"]
    print(f"{m:10s} {r['macro_f1']:8.4f} {bd['CL']['macro_f1']:7.4f} {bd['IT']['macro_f1']:7.4f} "
          f"{r['macro_p']:7.4f} {r['macro_r']:7.4f} {o['minutes']:5.1f} {o['peak_gb']:5.1f}")

In [ ]:
STAGE1_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2/student_kd_stage1/best.pt"
TEACHER_RR_CKPT = f"{P3_DIR}/rr_gqa_full_document_best.pt"
for p in (STAGE1_CKPT, TEACHER_RR_CKPT): assert os.path.exists(p), f"missing: {p}"
DEBUG_MODE = False   # True first, then False
STUDENT_CFG = dict(d_model=576, n_layer=7, n_head=12, n_kv_head=4)

def build_lm(cfg):
    return CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), max_seq_len=2048, tie_weights=True, **cfg)

# ---- teacher: the fine-tuned RR model ----
teacher = RRModel(build_lm(dict(d_model=768, n_layer=10, n_head=12, n_kv_head=4)), 768, NC).to(device)
teacher.load_state_dict(torch.load(TEACHER_RR_CKPT, map_location="cpu"), strict=False)
teacher.eval()
tres, _ = evaluate(teacher, val_docs, "document")
print("Teacher (loaded) dev:"); print_rr(tres)       # must match the earlier 0.5373, otherwise loading went wrong
assert abs(tres["macro_f1"] - 0.5373) < 0.01, "teacher did not load correctly"

def build_student():
    lm_ = build_lm(STUDENT_CFG)
    ck = torch.load(STAGE1_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)    # stage-1 student: random init, LM-distilled from the teacher
    return RRModel(lm_, STUDENT_CFG["d_model"], NC).to(device)

n_t = sum(p.numel() for p in teacher.lm.parameters()); n_s = sum(p.numel() for p in build_student().lm.parameters())
print(f"Backbone params: teacher {n_t/1e6:.1f}M | student {n_s/1e6:.1f}M ({100*n_s/n_t:.0f}%)")

STU = {}
for tag, alpha in [("kd", 0.5), ("nokd", 0.0)]:
    stu = build_student()
    out = train_rr(stu, f"student_{tag}", mode="document", teacher=teacher if alpha > 0 else None,
                   alpha=alpha, T=2.0, epochs=20, patience=5, backbone_lr=3e-4, head_lr=1e-3, debug=DEBUG_MODE)
    STU[tag] = out
    if not DEBUG_MODE: write_test_predictions(stu, f"student_{tag}_document", "document")
    del stu; torch.cuda.empty_cache()

print(f"\n{'model':16s} {'F1':>7s} {'CL':>7s} {'IT':>7s} {'P':>7s} {'R':>7s} {'params':>9s} {'min':>5s} {'GB':>5s}")
r = tres; bd = r["by_domain"]
print(f"{'teacher (full)':16s} {r['macro_f1']:7.4f} {bd['CL']['macro_f1']:7.4f} {bd['IT']['macro_f1']:7.4f} {r['macro_p']:7.4f} {r['macro_r']:7.4f} {n_t/1e6:8.1f}M {'-':>5s} {'-':>5s}")
for tag, o in STU.items():
    r = o["res"]; bd = r["by_domain"]
    print(f"{'student '+tag:16s} {r['macro_f1']:7.4f} {bd['CL']['macro_f1']:7.4f} {bd['IT']['macro_f1']:7.4f} {r['macro_p']:7.4f} {r['macro_r']:7.4f} {n_s/1e6:8.1f}M {o['minutes']:5.1f} {o['peak_gb']:5.1f}")

Teacher (loaded) dev:
ALL  macro-F1 0.5373 | P 0.5606 | R 0.5277 | acc 0.5630
  CL  macro-F1 0.4650 | P 0.4574 | R 0.5117 | acc 0.5408
  IT  macro-F1 0.5700 | P 0.6307 | R 0.5450 | acc 0.6046
Backbone params: teacher 87.5M | student 43.2M (49%)
[student_kd] mode=document windows=892 trainable=43,228,237 KD alpha=0.50 T=2.0
  ep 1 loss 4.2595 | val macro-F1 0.3708 CL 0.323 IT 0.407 | 0.3 min
  ep 2 loss 2.7617 | val macro-F1 0.3684 CL 0.310 IT 0.419 | 0.5 min
  ep 3 loss 1.7398 | val macro-F1 0.4007 CL 0.304 IT 0.437 | 0.8 min
  ep 4 loss 0.9599 | val macro-F1 0.4810 CL 0.386 IT 0.492 | 1.0 min
  ep 5 loss 0.6236 | val macro-F1 0.4440 CL 0.345 IT 0.512 | 1.3 min
  ep 6 loss 0.4504 | val macro-F1 0.4530 CL 0.380 IT 0.480 | 1.5 min
  ep 7 loss 0.2920 | val macro-F1 0.4037 CL 0.316 IT 0.481 | 1.8 min
  ep 8 loss 0.2146 | val macro-F1 0.4128 CL 0.396 IT 0.438 | 2.0 min
  ep 9 loss 0.1665 | val macro-F1 0.4256 CL 0.354 IT 0.470 | 2.3 min
  early stop
[student_kd] BEST on dev:
ALL  macro-F1 0

In [ ]:
# --- longer-window context modes (GQA supports up to 2048 positions) ---
CTX["doc1536"] = dict(max_sents=10**9, stride_frac=0.75, max_tokens=1536); BW["doc1536"] = 5
CTX["doc2048"] = dict(max_sents=10**9, stride_frac=0.75, max_tokens=2048); BW["doc2048"] = 4

DEBUG_MODE = False    # True first, then False
assert "build_gqa_rr" in globals(), "run the teacher cell first (it defines build_gqa_rr)"
for mode in ["doc1536", "doc2048"]:
    W = all_windows(train_docs, mode); assert max(b - a for _, a, b in W) > 0
    print(mode, "windows:", len(W))

LONG = {}
for mode in (["doc2048"] if DEBUG_MODE else ["doc1536", "doc2048"]):
    model = build_gqa_rr()
    out = train_rr(model, f"gqa_full_{mode}", mode=mode, epochs=10, patience=3,
                   backbone_lr=1e-4, head_lr=1e-3, debug=DEBUG_MODE)
    LONG[mode] = out
    if not DEBUG_MODE: write_test_predictions(model, f"gqa_full_{mode}", mode)
    del model; torch.cuda.empty_cache()

print(f"\n{'mode':10s} {'F1(all)':>8s} {'CL':>7s} {'IT':>7s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
print(f"{'document':10s} {0.5373:8.4f} {0.4650:7.4f} {0.5700:7.4f} {0.5606:7.4f} {0.5277:7.4f}   (1024, earlier run)")
for m, o in LONG.items():
    r = o["res"]; bd = r["by_domain"]
    print(f"{m:10s} {r['macro_f1']:8.4f} {bd['CL']['macro_f1']:7.4f} {bd['IT']['macro_f1']:7.4f} "
          f"{r['macro_p']:7.4f} {r['macro_r']:7.4f} {o['minutes']:5.1f} {o['peak_gb']:5.1f}")

doc1536 windows: 598
doc2048 windows: 455
[gqa_full_doc1536] mode=doc1536 windows=598 trainable=87,526,669 
  ep 1 loss 2.2291 | val macro-F1 0.4089 CL 0.334 IT 0.413 | 0.3 min
  ep 2 loss 0.9495 | val macro-F1 0.4577 CL 0.451 IT 0.473 | 0.7 min
  ep 3 loss 0.4158 | val macro-F1 0.5023 CL 0.412 IT 0.512 | 1.0 min
  ep 4 loss 0.1855 | val macro-F1 0.4911 CL 0.410 IT 0.549 | 1.3 min
  ep 5 loss 0.1156 | val macro-F1 0.5376 CL 0.503 IT 0.533 | 1.7 min
  ep 6 loss 0.0561 | val macro-F1 0.5025 CL 0.466 IT 0.484 | 2.0 min
  ep 7 loss 0.0199 | val macro-F1 0.5022 CL 0.470 IT 0.506 | 2.4 min
  ep 8 loss 0.0121 | val macro-F1 0.5277 CL 0.456 IT 0.547 | 2.7 min
  early stop
[gqa_full_doc1536] BEST on dev:
ALL  macro-F1 0.5376 | P 0.5832 | R 0.5411 | acc 0.5728
  CL  macro-F1 0.5027 | P 0.5034 | R 0.5416 | acc 0.5630
  IT  macro-F1 0.5329 | P 0.6298 | R 0.5048 | acc 0.5912
Saved prediction_gqa_full_doc1536_{all,CL,IT}.json (10 docs, 2197 sentences)
[gqa_full_doc2048] mode=doc2048 windows=455 trai

In [ ]:
import math
USE_POS, POS_SCALE, POS_DIM = True, 3.0, 10

def collate(wins, docs):
    seqs, spans, ys = [], [], []
    for di, a, b in wins:
        d = docs[di]; n = len(d["sent_ids"]); ids, sp = [], []
        for k in range(a, b):
            s = len(ids); ids += d["sent_ids"][k]
            sp.append((s, len(ids), k / max(n - 1, 1), math.log(n) / 8))     # + relative position, doc length
        seqs.append(ids); spans.append(sp)
        if d["y"] is not None: ys += d["y"][a:b]
    L = max(len(s) for s in seqs)
    x = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    for j, s in enumerate(seqs): x[j, :len(s)] = torch.tensor(s)
    return x, spans, (torch.tensor(ys) if ys else None)

def pool(H, spans):
    jw, st, en, rr, ll = [], [], [], [], []
    for j, sp in enumerate(spans):
        for s, e, r, l in sp: jw.append(j); st.append(s); en.append(e); rr.append(r); ll.append(l)
    jw, st, en = [torch.tensor(t, device=H.device) for t in (jw, st, en)]
    last = H[jw, en - 1]
    cs = torch.cat([torch.zeros_like(H[:, :1]).float(), H.float().cumsum(1)], dim=1)
    mean = ((cs[jw, en] - cs[jw, st]) / (en - st).unsqueeze(1)).to(H.dtype)
    feats = [last, mean]
    if USE_POS:
        r = torch.tensor(rr, device=H.device, dtype=torch.float32)
        f = [r.unsqueeze(1)] + [fn(2 * math.pi * k * r).unsqueeze(1) for k in (1, 2, 3, 4) for fn in (torch.sin, torch.cos)] \
            + [torch.tensor(ll, device=H.device, dtype=torch.float32).unsqueeze(1)]
        feats.append((torch.cat(f, 1) * POS_SCALE).to(H.dtype))
    return torch.cat(feats, -1)

class RRModel(nn.Module):
    def __init__(self, lm, d_model, n_labels=NC, dropout=0.1):
        super().__init__()
        self.lm, self.drop = lm, nn.Dropout(dropout)
        self.head = nn.Linear(2 * d_model + (POS_DIM if USE_POS else 0), n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
    def hidden(self, ids):
        lm = self.lm; x = lm.token_embedding(ids)
        for layer in lm.layers: x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
        return lm.norm(x)
    def forward(self, ids, spans): return self.head(self.drop(pool(self.hidden(ids), spans)))

DEBUG_MODE = False     # True first, then False
model = build_gqa_rr()
out_pos = train_rr(model, "gqa_full_document_pos", mode="document", epochs=10, patience=3,
                   backbone_lr=1e-4, head_lr=1e-3, debug=DEBUG_MODE)
if not DEBUG_MODE: write_test_predictions(model, "gqa_full_document_pos", "document")

In [ ]:
assert "RR_RESULTS" in globals() and "document" in RR_RESULTS, "re-run the teacher cell for mode 'document' first"
devP = RR_RESULTS["document"]["val_probs"]
testP = pickle.load(open(f"{P3_DIR}/probs_test_gqa_full_document.pkl", "rb"))

def trans_matrix(docs, alpha=1.0):
    C = np.full((NC, NC), alpha)
    for d in docs:
        for a, b in zip(d["y"][:-1], d["y"][1:]): C[a, b] += 1
    return np.log(C / C.sum(1, keepdims=True))

def viterbi(logp, logA, lam):
    n = logp.shape[0]; dp = np.zeros((n, NC)); bp = np.zeros((n, NC), int); dp[0] = logp[0]
    for t in range(1, n):
        s = dp[t - 1][:, None] + lam * logA          # (prev, cur)
        bp[t] = s.argmax(0); dp[t] = s.max(0) + logp[t]
    path = [int(dp[-1].argmax())]
    for t in range(n - 1, 0, -1): path.append(int(bp[t][path[-1]]))
    return path[::-1]

logA = trans_matrix(train_docs)
def decode(P, docs, lam):
    return {d["id"]: (viterbi(np.log(P[d["id"]] + 1e-9), logA, lam) if lam > 0 else P[d["id"]].argmax(1).tolist()) for d in docs}

def score_dev(preds):
    G = {k: ([], []) for k in ("CL", "IT", "ALL")}
    for d in val_docs:
        for k in (d["domain"], "ALL"): G[k][0].extend(d["y"]); G[k][1].extend(preds[d["id"]])
    return {k: rr_scores(*G[k]) for k in G}

print(f"{'lambda':>7s} {'F1':>7s} {'CL':>7s} {'IT':>7s} {'P':>7s} {'R':>7s}")
best_lam, best_f1 = 0, -1
devP = out_pos["val_probs"]; testP = pickle.load(open(f"{P3_DIR}/probs_test_gqa_full_document_pos.pkl", "rb"))
for lam in [0, 0.5, 1, 2]:
    r = score_dev(decode(devP, val_docs, lam))
    print(f"lam {lam}: F1 {r['ALL']['macro_f1']:.4f} CL {r['CL']['macro_f1']:.4f} IT {r['IT']['macro_f1']:.4f}")
    if r["ALL"]["macro_f1"] > best_f1: best_lam, best_f1 = lam, r["ALL"]["macro_f1"]
print("best lambda:", best_lam)

pred = {d["id"]: [LABELS[i] for i in decode(testP, [d], best_lam)[d["id"]]] for d in test_docs}
for d in test_docs: assert len(pred[d["id"]]) == len(d["text"])
for suf, sel in [("all", pred), ("CL", {d["id"]: pred[d["id"]] for d in test_docs if d["domain"] == "CL"}),
                 ("IT", {d["id"]: pred[d["id"]] for d in test_docs if d["domain"] == "IT"})]:
    json.dump(sel, open(f"{P3_DIR}/prediction_gqa_full_document_viterbi_{suf}.json", "w"), ensure_ascii=False)
print("saved prediction_gqa_full_document_viterbi_{all,CL,IT}.json")

 lambda      F1      CL      IT       P       R
lam 0: F1 0.5290 CL 0.4269 IT 0.5452
lam 0.5: F1 0.5310 CL 0.4262 IT 0.5467
lam 1: F1 0.5431 CL 0.4493 IT 0.5402
lam 2: F1 0.5423 CL 0.5182 IT 0.5367
best lambda: 1
saved prediction_gqa_full_document_viterbi_{all,CL,IT}.json


In [ ]:
import itertools
assert "decode" in globals() and "score_dev" in globals() and "logA" in globals(), "run the earlier Viterbi cell first (it defines these)"

cands = {}
def add(tag, getter, pkl):
    try:
        dv = getter(); tp = pickle.load(open(f"{P3_DIR}/probs_test_{pkl}.pkl", "rb"))
        assert set(dv) == {d["id"] for d in val_docs}, "dev probs are from a debug run"
        cands[tag] = (dv, tp)
    except Exception as e: print(f"skip {tag}: {type(e).__name__}")
add("doc1024", lambda: RR_RESULTS["document"]["val_probs"], "gqa_full_document")
add("local8",  lambda: RR_RESULTS["local8"]["val_probs"],   "gqa_full_local8")
add("sentence",lambda: RR_RESULTS["sentence"]["val_probs"], "gqa_full_sentence")
add("pos",     lambda: out_pos["val_probs"],                "gqa_full_document_pos")
add("doc1536", lambda: LONG["doc1536"]["val_probs"],        "gqa_full_doc1536")
add("doc2048", lambda: LONG["doc2048"]["val_probs"],        "gqa_full_doc2048")
add("stu_kd",  lambda: STU["kd"]["val_probs"],              "student_kd_document")
add("stu_nokd",lambda: STU["nokd"]["val_probs"],            "student_nokd_document")
names = list(cands); print("available:", names)

def avg(ps): return {k: np.mean([p[k] for p in ps], 0) for k in ps[0]}
def dev_f1(sub, lam):
    return score_dev(decode(avg([cands[n][0] for n in sub]), val_docs, lam))["ALL"]["macro_f1"]

for n in names: print(f"  single {n:9s} dev F1 {dev_f1([n], 0):.4f}")
rows = sorted(((dev_f1(s, 0), s) for k in range(2, len(names) + 1) for s in itertools.combinations(names, k)), reverse=True)
print(f"\nall {len(names)} models: lam0 {dev_f1(names, 0):.4f} | lam1 {dev_f1(names, 1):.4f}")
print("top subsets (dev, selected on 10 docs so slightly optimistic):")
for f, s in rows[:6]: print(f"  {f:.4f} (lam1 {dev_f1(s, 1):.4f})  {s}")

def write_ens(sub, lam, tag):
    P = avg([cands[n][1] for n in sub]); pr = decode(P, test_docs, lam)
    pred = {d["id"]: [LABELS[i] for i in pr[d["id"]]] for d in test_docs}
    for d in test_docs: assert len(pred[d["id"]]) == len(d["text"])
    for suf, sel in [("all", pred), ("CL", {d["id"]: pred[d["id"]] for d in test_docs if d["domain"] == "CL"}),
                     ("IT", {d["id"]: pred[d["id"]] for d in test_docs if d["domain"] == "IT"})]:
        json.dump(sel, open(f"{P3_DIR}/prediction_{tag}_{suf}.json", "w"), ensure_ascii=False)
    print("saved", tag, "<-", sub, f"lam={lam}")

best = rows[0][1]
for lam in (0, 1):
    write_ens(names, lam, f"ens_all_l{lam}")
    write_ens(best, lam, f"ens_best_l{lam}")

available: ['doc1024', 'local8', 'sentence', 'pos', 'doc1536', 'doc2048', 'stu_kd', 'stu_nokd']
  single doc1024   dev F1 0.5373
  single local8    dev F1 0.4402
  single sentence  dev F1 0.3497
  single pos       dev F1 0.5290
  single doc1536   dev F1 0.5376
  single doc2048   dev F1 0.5614
  single stu_kd    dev F1 0.4810
  single stu_nokd  dev F1 0.4773

all 8 models: lam0 0.5564 | lam1 0.5283
top subsets (dev, selected on 10 docs so slightly optimistic):
  0.5718 (lam1 0.5413)  ('doc1024', 'doc2048', 'stu_nokd')
  0.5701 (lam1 0.5594)  ('doc1536', 'doc2048', 'stu_kd', 'stu_nokd')
  0.5688 (lam1 0.5631)  ('doc1024', 'doc2048', 'stu_kd')
  0.5684 (lam1 0.5380)  ('doc1024', 'local8', 'doc1536', 'doc2048', 'stu_kd')
  0.5678 (lam1 0.5551)  ('doc1024', 'doc1536', 'doc2048', 'stu_kd')
  0.5670 (lam1 0.5302)  ('doc2048', 'stu_nokd')
saved ens_all_l0 <- ['doc1024', 'local8', 'sentence', 'pos', 'doc1536', 'doc2048', 'stu_kd', 'stu_nokd'] lam=0
saved ens_best_l0 <- ('doc1024', 'doc2048', 's